# 03 — Train CV Models (Phase 5)

Trains the small custom CNN on one image representation at a time (waveform / STFT / CWT), per the plan's instruction to compare representations independently before combining them. Uses an NVIDIA CUDA GPU when available (falls back to CPU otherwise).

Run the whole notebook once per representation by changing `REPRESENTATION` in the config cell and re-running from there down — that gives you three checkpoints and three metric histories to compare.

*Plain (unweighted) BCE loss is used deliberately here — Phase 6/7 is where class-imbalance handling (weighted loss vs. weighted sampling, augmentation) gets compared properly on top of whichever representation wins this comparison.*

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import average_precision_score, balanced_accuracy_score, recall_score

root = Path.cwd()
while not (root / "config.yaml").exists():
    root = root.parent
sys.path.insert(0, str(root))

from src.datasets import ApneaImageDataset
from src.models import MODEL_REGISTRY

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    print(f"using device: {device} ({torch.cuda.get_device_name(0)})")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"using device: {device}")
else:
    device = torch.device("cpu")
    print(f"using device: {device}")

PIN_MEMORY = device.type == "cuda"

## Config — change these and re-run from here to compare representations/models

In [ ]:
MODEL_NAME = "small_cnn"          # "small_cnn" or "efficientnet_b0"
REPRESENTATION = "waveform"       # "waveform", "stft", or "cwt"
EPOCHS = 8
BATCH_SIZE = 64
LR = 1e-3
USE_AMP = device.type == "cuda"   # mixed precision on the RTX 3060's tensor cores

CHECKPOINT_DIR = root / "models" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## Load data

In [ ]:
train_ds = ApneaImageDataset("train", REPRESENTATION)
val_ds = ApneaImageDataset("val", REPRESENTATION)
print(f"train windows: {len(train_ds)}, val windows: {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)

## Model, optimizer, loss

In [5]:
model = MODEL_REGISTRY[MODEL_NAME]().to(device)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
criterion = torch.nn.BCEWithLogitsLoss()

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{MODEL_NAME}: {n_params:,} params ({n_trainable:,} trainable)")

small_cnn: 60,641 params (60,641 trainable)


## Evaluation helper

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=PIN_MEMORY)
            labels = labels.to(device, non_blocking=PIN_MEMORY)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(images)
            probs = torch.sigmoid(logits.float())
            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())
    y_true = torch.cat(all_labels).numpy()
    y_prob = torch.cat(all_probs).numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "sensitivity": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "specificity": float(recall_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

## Training loop

Saves the checkpoint with the best validation PR-AUC to `models/checkpoints/{model}_{representation}.pt`.

In [ ]:
checkpoint_path = CHECKPOINT_DIR / f"{MODEL_NAME}_{REPRESENTATION}.pt"
best_pr_auc = -1.0
history = []
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_ds)
    val_metrics = evaluate(model, val_loader)
    elapsed = time.time() - t0
    print(
        f"epoch {epoch}/{EPOCHS} train_loss={train_loss:.4f} "
        f"val_pr_auc={val_metrics['pr_auc']:.4f} val_balanced_acc={val_metrics['balanced_accuracy']:.4f} "
        f"({elapsed:.1f}s, {len(train_ds) / elapsed:,.0f} windows/s)"
    )
    history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

    if val_metrics["pr_auc"] > best_pr_auc:
        best_pr_auc = val_metrics["pr_auc"]
        torch.save(model.state_dict(), checkpoint_path)

print(f"\nbest val PR-AUC: {best_pr_auc:.4f} -> {checkpoint_path}")

## Plot training curves

In [ ]:
epochs_x = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_x, [h["train_loss"] for h in history], marker="o")
axes[0].set_title("train loss")
axes[0].set_xlabel("epoch")

axes[1].plot(epochs_x, [h["pr_auc"] for h in history], marker="o", label="PR-AUC")
axes[1].plot(epochs_x, [h["balanced_accuracy"] for h in history], marker="o", label="balanced acc")
axes[1].set_title(f"validation metrics ({REPRESENTATION})")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## Save this run's metrics for later comparison

Writes `reports/metrics/{model}_{representation}.json`. After running this notebook once per representation, load all three JSON files to compare which representation actually carries signal — the empirical question Phase 5 exists to answer.

In [ ]:
import json

metrics_dir = root / "reports" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)
with open(metrics_dir / f"{MODEL_NAME}_{REPRESENTATION}.json", "w") as f:
    json.dump({"model": MODEL_NAME, "representation": REPRESENTATION, "history": history}, f, indent=2)
print(f"wrote {metrics_dir / f'{MODEL_NAME}_{REPRESENTATION}.json'}")

## Compare representations (run after training all three)

In [ ]:
comparison = {}
for rep in ["waveform", "stft", "cwt"]:
    path = metrics_dir / f"{MODEL_NAME}_{rep}.json"
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        best = max(data["history"], key=lambda h: h["pr_auc"])
        comparison[rep] = best

for rep, best in comparison.items():
    print(f"{rep:10s} best_pr_auc={best['pr_auc']:.4f} balanced_acc={best['balanced_accuracy']:.4f} "
          f"sensitivity={best['sensitivity']:.4f} specificity={best['specificity']:.4f}")